# Test Gradient Flow Through Simulation

This notebook tests whether gradients can flow backward through multiple time steps of ODE simulation.

We want to verify that:
1. Gradients flow from the final state back to initial parameters
2. Gradients flow through multiple simulation steps
3. The deepcopy issue is properly diagnosed

In [ ]:
import torch
import sys
sys.path.insert(0, '../src')

from rpasim.ode.ab import AB
from rpasim.env.base import DifferentiableEnv

## Setup: Simple AB ODE

We'll use a simple AB ODE with differentiable parameters.

In [ ]:
# Create ODE with differentiable parameters
alphas = torch.tensor([1.0, 0.5, 0.5], requires_grad=True)
betas = torch.tensor([1.0, 1.0])

ode = AB(
    differentiable_params=alphas,
    fixed_params=betas,
)

print(f"Initial alphas: {alphas}")
print(f"alphas.requires_grad: {alphas.requires_grad}")
print(f"alphas.grad: {alphas.grad}")

## Test 5: Computational Graph Analysis

Let's directly inspect the computational graph to verify connectivity.

In [ ]:
def check_alphas_in_graph(reward_tensor, alphas):
    """Check if alphas tensor appears in the computational graph."""
    grads = torch.autograd.grad(reward_tensor, alphas, retain_graph=True, allow_unused=True)
    
    if grads[0] is None:
        return False, "None"
    elif grads[0].abs().sum() == 0:
        return False, "Zero"
    else:
        return True, grads[0].abs().sum().item()

# Reward function
def reward_fn(state):
    target_b = 1.0
    return -(state[1] - target_b) ** 2

initial_state = torch.tensor([1.0, 0.5])

print("Setup complete")

### Test 5a: WITHOUT Fix - Check Each Step

In [ ]:
# Create fresh parameters
alphas = torch.tensor([1.0, 0.5, 0.5], requires_grad=True)
betas = torch.tensor([1.0, 1.0])

ode = AB(differentiable_params=alphas, fixed_params=betas)
env = DifferentiableEnv(
    initial_ode=ode,
    reward_fn=reward_fn,
    initial_state=initial_state,
    time_horizon=10.0,
    n_reward_steps=100,
)

obs, info = env.reset()
current_ode, state = obs
current_ode.differentiable_params = alphas

n_steps = 3
time_per_step = 10.0 / n_steps
rewards = []

for step in range(n_steps):
    print(f"\nStep {step + 1} (WITHOUT reconnecting):")
    
    obs, reward, terminated, truncated, info = env.step((current_ode, time_per_step))
    current_ode, state = obs
    
    # NOT reconnecting!
    
    rewards.append(reward)
    connected, info = check_alphas_in_graph(reward, alphas)
    print(f"  Connected to alphas: {connected}")
    if connected:
        print(f"  Gradient magnitude: {info:.3e}")
    else:
        print(f"  Status: {info}")

total_reward = sum(rewards)
connected, info = check_alphas_in_graph(total_reward, alphas)
print(f"\nTotal reward connected: {connected}")
if connected:
    grad = torch.autograd.grad(total_reward, alphas, retain_graph=True)[0]
    print(f"Total gradient: {grad}")

### Test 5b: WITH Fix - Check Each Step

In [ ]:
# Create fresh parameters
alphas = torch.tensor([1.0, 0.5, 0.5], requires_grad=True)
betas = torch.tensor([1.0, 1.0])

ode = AB(differentiable_params=alphas, fixed_params=betas)
env = DifferentiableEnv(
    initial_ode=ode,
    reward_fn=reward_fn,
    initial_state=initial_state,
    time_horizon=10.0,
    n_reward_steps=100,
)

obs, info = env.reset()
current_ode, state = obs
current_ode.differentiable_params = alphas

n_steps = 3
time_per_step = 10.0 / n_steps
rewards = []

for step in range(n_steps):
    print(f"\nStep {step + 1} (WITH reconnecting):")
    
    obs, reward, terminated, truncated, info = env.step((current_ode, time_per_step))
    current_ode, state = obs
    
    # FIX: Reconnect!
    current_ode.differentiable_params = alphas
    
    rewards.append(reward)
    connected, info = check_alphas_in_graph(reward, alphas)
    print(f"  Connected to alphas: {connected}")
    if connected:
        print(f"  Gradient magnitude: {info:.3e}")
    else:
        print(f"  Status: {info}")

total_reward = sum(rewards)
connected, info = check_alphas_in_graph(total_reward, alphas)
print(f"\nTotal reward connected: {connected}")
if connected:
    grad = torch.autograd.grad(total_reward, alphas, retain_graph=True)[0]
    print(f"Total gradient: {grad}")

## Conclusion

The computational graph analysis proves:

**WITHOUT fix:** Only Step 1 is connected to alphas (Steps 2 & 3 are disconnected by deepcopy)

**WITH fix:** All 3 steps are connected to alphas

This demonstrates the gradient differences are due to missing graph connections, not randomness.